# Fase 3 — Preparación de los Datos
## Pipeline de Consolidación, Limpieza e Integración Macroeconómica

**Notebook:** `notebooks/02_preparacion_datos.ipynb`  
**Responsable:** Kukis | **Apoyo:** Steve, Sofía  
**Objetivo:** Consolidar los datasets del Grupo A (vivienda), aplicar filtros de calidad y lógica temporal (2019-2024), corregir inconsistencias monetarias y de nombres de ciudades, deduplicar ofertas utilizando coordenadas geográficas (evitando el colapso masivo), imputar valores nulos, unificar y fusionar con datos macroeconómicos del Grupo B, construir variables derivadas de accesibilidad y exportar el dataset limpio y verificado.

## 1. Configuración Inicial e Importaciones

Importación de librerías, configuración de directorios robustos a la ubicación de ejecución, y definición del esquema canónico básico y funciones de registro métrico.

In [ ]:
import pandas as pd
import numpy as np
import os
import re

# Directorios robustos a la ubicación de ejecución
if os.path.exists(os.path.join("data", "raw")):
    DIR_RAW = os.path.join("data", "raw")
    DIR_PROCESSED = os.path.join("data", "processed")
else:
    DIR_RAW = os.path.join("..", "data", "raw")
    DIR_PROCESSED = os.path.join("..", "data", "processed")
os.makedirs(DIR_PROCESSED, exist_ok=True)

COLS_CANONICAS = [
    'price', 'area', 'rooms', 'bathrooms', 'property_type', 
    'city', 'lat', 'lon', 'created_on', 'barrio', 'parking', 'estrato', 'fuente'
]

# Diccionario para el reporte de limpieza
reporte_metricas = []

def registrar_metrica(paso, operacion, df_in, df_out):
    regs_in = len(df_in) if df_in is not None else 0
    regs_out = len(df_out)
    eliminados = regs_in - regs_out if df_in is not None else 0
    pct = (eliminados / regs_in * 100) if regs_in > 0 else 0

    # Distribución por fuente en la salida (para diagnóstico)
    dist_fuentes = {}
    if df_out is not None and 'fuente' in df_out.columns:
        dist_fuentes = df_out.groupby('fuente').size().to_dict()

    reporte_metricas.append({
        'Paso': paso,
        'Operacion': operacion,
        'Regs_Entrada': regs_in,
        'Regs_Salida': regs_out,
        'Regs_Eliminados': eliminados,
        'Pct_Eliminado': round(pct, 2),
        'Dist_Fuentes': str(dist_fuentes)
    })

def diagnostico_por_fuente(df, nombre_paso):
    """Imprime cuántos registros quedan de cada fuente después de cada paso."""
    print(f"\n--- Diagnóstico por fuente: {nombre_paso} ---")
    dist = df.groupby('fuente').size().sort_values(ascending=False)
    for fuente, n in dist.items():
        print(f"  {fuente}: {n:,}")
    print(f"  TOTAL: {len(df):,}")

## 2. Consolidación de Datasets (Sección 1)

Carga de todas las fuentes heterogéneas, mapeo y renombramiento al esquema canónico y unificación en una única base de datos consolidada.

In [ ]:
def cargar_y_canonizar_datasets():
    datasets = []
    
    # A1: Properati Colombia
    df1 = pd.read_csv(os.path.join(DIR_RAW, "A1_colombia_housing_properties.csv"), encoding='utf-8-sig', low_memory=False)
    if 'operation_type' in df1.columns:
        df1 = df1[df1['operation_type'] == 'Venta'].copy()
    if 'l1' in df1.columns:
        df1 = df1[df1['l1'] == 'Colombia'].copy()
    
    # Avoid duplicate created_on
    if 'created_on' in df1.columns and 'start_date' in df1.columns:
        df1 = df1.drop(columns=['created_on'])
        
    rename_dict1 = {
        'bedrooms': 'rooms',
        'bathrooms': 'bathrooms', 'property_type': 'property_type',
        'l3': 'city', 'start_date': 'created_on'
    }
    if 'surface_total' in df1.columns:
        rename_dict1['surface_total'] = 'area'
    elif 'area' in df1.columns:
        rename_dict1['area'] = 'area'
        
    df1 = df1.rename(columns=rename_dict1)
    df1['fuente'] = 'A1_Properati'
    datasets.append(df1)
    
    # A2: FincaRaiz Colombia 2023-2024 (B1 Fix: No multiplicar por 1_000_000)
    df2 = pd.read_csv(os.path.join(DIR_RAW, "A2_fincaraiz_colombia.csv"), encoding='utf-8-sig')
    df2.columns = [c.replace('\ufeff', '') for c in df2.columns]
    
    df2 = df2.rename(columns={
        'Precio': 'price', 'Area Construida': 'area', 'Habitaciones': 'rooms',
        'Banos': 'bathrooms', 'Tipo Propiedad': 'property_type', 'Ciudad': 'city',
        'Estrato': 'estrato', 'Fecha Actualizacion': 'created_on'
    })
    
    if 'price' in df2.columns:
        df2['price'] = pd.to_numeric(df2['price'].astype(str).str.replace(r'[^\d.]', '', regex=True), errors='coerce')
        # B1 Fix: El precio en A2 ya está en COP completos. NO multiplicar por 1_000_000.
    

    # Extraer coordenadas geográficas de FincaRaiz
    if 'Link Google Maps' in df2.columns:
        coordenadas = df2['Link Google Maps'].str.extract(r'q=([\d.-]+),([\d.-]+)')
        df2['lat'] = pd.to_numeric(coordenadas[0], errors='coerce')
        df2['lon'] = pd.to_numeric(coordenadas[1], errors='coerce')
    df2['fuente'] = 'A2_FincaRaiz_Kaggle'
    datasets.append(df2)
    
    # A3: Colombia House Prediction (B2 Fix: Asignar city = 'Bogotá', e inferir property_type)
    df3 = pd.read_csv(os.path.join(DIR_RAW, "A3_colombia_house_prediction.csv"), encoding='utf-8-sig')
    df3.columns = [c.replace('\ufeff', '') for c in df3.columns]
    df3 = df3.rename(columns={
        'valor': 'price', 'area': 'area', 'habitaciones': 'rooms',
        'banos': 'bathrooms', 'estrato': 'estrato',
        'latitud': 'lat', 'longitud': 'lon'
    })
    df3['city'] = 'Bogotá'
    # Definir property_type usando heurística basada en numeroascensores y area
    df3['property_type'] = np.where(df3['numeroascensores'].fillna(0) > 0, 'Apartamento', 
                                    np.where(df3['area'] > 160, 'Casa', 'Apartamento'))
    df3['fuente'] = 'A3_Kaggle'
    datasets.append(df3)
    
    # A4: Real Estate Bogotá
    df4 = pd.read_csv(os.path.join(DIR_RAW, "A4_real_estate_bogota.csv"), encoding='latin1')
    df4.columns = [c.replace('\ufeff', '') for c in df4.columns]
    
    rename_a4 = {}
    for c in df4.columns:
        if 'Valor' in c: rename_a4[c] = 'price'
        elif 'rea' in c: rename_a4[c] = 'area'
        elif 'Habitaciones' in c: rename_a4[c] = 'rooms'
        elif 'Ba' in c and 'os' in c: rename_a4[c] = 'bathrooms'
        elif 'Tipo' in c: rename_a4[c] = 'property_type'
    
    df4 = df4.rename(columns=rename_a4)
    if 'price' in df4.columns:
        df4['price'] = pd.to_numeric(df4['price'].astype(str).str.replace(r'[^\d.]', '', regex=True), errors='coerce')
    df4['city'] = 'Bogotá'
    df4['fuente'] = 'A4_Bogota_Kaggle'
    datasets.append(df4)
    
    # A5: Medellín Properties 2023
    df5 = pd.read_csv(os.path.join(DIR_RAW, "A5_medellin_properties_2023.csv"), encoding='utf-8-sig')
    df5.columns = [c.replace('\ufeff', '') for c in df5.columns]
    df5 = df5.rename(columns={
        'price': 'price', 'area': 'area', 'rooms': 'rooms',
        'baths': 'bathrooms', 'property_type': 'property_type', 
        'neighbourhood': 'barrio', 'stratum': 'estrato'
    })
    df5['city'] = 'Medellín'
    df5['fuente'] = 'A5_Medellin_Kaggle'
    datasets.append(df5)
    
    # A6: Real Estate Bogotá 2023
    df6 = pd.read_csv(os.path.join(DIR_RAW, "A6_real_estate_bogota_2023.csv"), encoding='latin1')
    df6.columns = [c.replace('\ufeff', '') for c in df6.columns]
    
    rename_a6 = {}
    for c in df6.columns:
        if 'precio' in c.lower(): rename_a6[c] = 'price'
        elif 'rea' in c.lower() or 'area' in c.lower(): rename_a6[c] = 'area'
        elif 'habitaciones' in c.lower() or 'cuartos' in c.lower(): rename_a6[c] = 'rooms'
        elif 'ba' in c.lower() and 'os' in c.lower(): rename_a6[c] = 'bathrooms'
        elif 'tipo_de_inmueble' in c.lower(): rename_a6[c] = 'property_type'
    
    df6 = df6.rename(columns=rename_a6)
    df6['city'] = 'Bogotá'
    df6['fuente'] = 'A6_Bogota2023_Kaggle'
    datasets.append(df6)
    
    # A7: Villavicencio Scraping (B3 Fix: renombrar columnas y filtrar por Venta/Villavicencio)
    if os.path.exists(os.path.join(DIR_RAW, "A7_fincaraiz_villavicencio_scraping.csv")):
        df7 = pd.read_csv(os.path.join(DIR_RAW, "A7_fincaraiz_villavicencio_scraping.csv"), encoding='utf-8-sig')
        df7 = df7.rename(columns={
            'precio_cop': 'price', 'area_m2': 'area', 'habitaciones': 'rooms',
            'banos': 'bathrooms', 'tipo_inmueble': 'property_type', 'ciudad': 'city',
            'fecha_scraping': 'created_on'
        })
        if 'tipo_operacion' in df7.columns:
            df7 = df7[df7['tipo_operacion'] == 'Venta'].copy()
        df7 = df7[df7['city'].astype(str).str.lower().str.contains('villavicencio', na=False)].copy()
        df7['fuente'] = 'A7_Scraping_Villavicencio'
        datasets.append(df7)
    
    # A8: Características precios vivienda nueva Bogotá UPZ (B4 Fix: renombrar columnas y asignar city/prop_type)
    if os.path.exists(os.path.join(DIR_RAW, "A8_carac_pre_viv_nueva.csv")):
        df8 = pd.read_csv(os.path.join(DIR_RAW, "A8_carac_pre_viv_nueva.csv"), encoding='utf-8-sig')
        df8 = df8.rename(columns={
            'precios': 'price', 'area': 'area', 'alcobas': 'rooms', 'baños': 'bathrooms'
        })
        df8['city'] = 'Bogotá'
        df8['property_type'] = 'Apartamento'
        df8['fuente'] = 'A8_CaracPreVivNueva'
        datasets.append(df8)
    
    df_lista_filtrada = []
    for df in datasets:
        for col in COLS_CANONICAS:
            if col not in df.columns:
                df[col] = np.nan
        df_lista_filtrada.append(df[COLS_CANONICAS])
        
    df_consolidado = pd.concat(df_lista_filtrada, ignore_index=True)
    return df_consolidado

print("1. Cargando y canonizando datasets...")
df_raw_consolidado = cargar_y_canonizar_datasets()
registrar_metrica(0, "Consolidacion Inicial", None, df_raw_consolidado)
print(f"Total registros cargados: {len(df_raw_consolidado)}")

## 3. Limpieza de Precios, Monedas e Invalidez Física (Sección 2)

Conversión de precios en USD a COP mediante TRM histórica, cálculo de precios por m² cuando corresponde y eliminación de precios atípicos y áreas físicamente imposibles (menores a 15m² o mayores a 800m², preservando los nulos).

In [ ]:
TRM_HISTORICA = {
    2015: 2746.0, 2016: 3051.0, 2017: 2951.0, 2018: 2956.0, 2019: 3281.0,
    2020: 3693.0, 2021: 3743.0, 2022: 4256.0, 2023: 4325.0, 2024: 4000.0
}

def limpiar_precios_y_monedas(df):
    df = df.copy()

    df['created_on'] = pd.to_datetime(df['created_on'], errors='coerce')
    df['year_temp'] = df['created_on'].dt.year.fillna(2023).astype(int)

    is_properati = df['fuente'] == 'A1_Properati'

    if 'currency' in df.columns:
        is_usd = df['currency'] == 'USD'
        for yr, trm in TRM_HISTORICA.items():
            mask = is_properati & is_usd & (df['year_temp'] == yr)
            df.loc[mask, 'price'] = df.loc[mask, 'price'] * trm

    is_cop_m2 = is_properati & (df['price'] < 1000000) & (df['price'] > 5000) & (df['area'] > 10)
    df.loc[is_cop_m2, 'price'] = df.loc[is_cop_m2, 'price'] * df.loc[is_cop_m2, 'area']

    df = df[df['price'].notnull()]
    df = df[(df['price'] >= 10000000) & (df['price'] <= 10000000000)]

    # Filtro de área: eliminar valores físicamente imposibles (Fase 2 recomienda 15–800 m²)
    # Los nulos se conservan — se imputan más adelante en la Sección 8
    df = df[(df['area'].isna()) | ((df['area'] >= 15) & (df['area'] <= 800))]

    df = df.drop(columns=['year_temp'])
    return df

print("2. Limpiando precios y monedas...")
df_clean_precios = limpiar_precios_y_monedas(df_raw_consolidado)
registrar_metrica(1, "Limpieza Precios e Invalidez", df_raw_consolidado, df_clean_precios)

## 4. Estandarización de Ciudades (Sección 3)

Mapeo sistemático de nombres de ciudades heterogéneas, sin tildes, con sufijos de departamento o variantes regionales, a un catálogo estandarizado unificado de 12 ciudades canónicas.

In [ ]:
MAPA_CIUDADES = {
    # Bogotá
    'bogota': 'Bogotá', 'bogotá': 'Bogotá',
    'santa fe de bogota': 'Bogotá', 'santa fe de bogotá': 'Bogotá',
    'bogota d.c.': 'Bogotá', 'bogotá d.c.': 'Bogotá',
    'bogota d. c.': 'Bogotá', 'bogotá d. c.': 'Bogotá',
    'bogota d.c': 'Bogotá', 'bogotá d.c': 'Bogotá',
    'distrito capital': 'Bogotá', 'bogota dc': 'Bogotá',
    # Medellín
    'medellin': 'Medellín', 'medelln': 'Medellín', 'medellín': 'Medellín',
    'medellin antioquia': 'Medellín',
    # Cali
    'cali': 'Cali', 'santiago de cali': 'Cali',
    'cali valle': 'Cali', 'cali valle del cauca': 'Cali',
    # Barranquilla
    'barranquilla': 'Barranquilla', 'barranq': 'Barranquilla',
    'barranquilla atlantico': 'Barranquilla',
    # Cartagena
    'cartagena': 'Cartagena', 'cartagena de indias': 'Cartagena',
    'cartagena bolivar': 'Cartagena',
    # Bucaramanga
    'bucaramanga': 'Bucaramanga', 'bucara': 'Bucaramanga',
    'bucaramanga santander': 'Bucaramanga',
    # Pereira
    'pereira': 'Pereira', 'pereira risaralda': 'Pereira',
    # Manizales
    'manizales': 'Manizales', 'manizales caldas': 'Manizales',
    # Armenia
    'armenia': 'Armenia', 'armenia quindio': 'Armenia', 'armenia quindío': 'Armenia',
    # Cúcuta
    'cucuta': 'Cúcuta', 'cúcuta': 'Cúcuta',
    'sanjose de cucuta': 'Cúcuta', 'san jose de cucuta': 'Cúcuta',
    'san josé de cúcuta': 'Cúcuta',
    # Ibagué
    'ibague': 'Ibagué', 'ibagué': 'Ibagué',
    # Villavicencio
    'villavicencio': 'Villavicencio', 'villavo': 'Villavicencio',
    'villavicencio meta': 'Villavicencio',
}

def estandarizar_ciudades(df):
    df = df.copy()
    df['city_raw'] = df['city'].astype(str).str.lower().str.strip()
    df['city_clean'] = df['city_raw'].map(MAPA_CIUDADES)
    df = df[df['city_clean'].notnull()].copy()
    df = df.drop(columns=['city', 'city_raw']).rename(columns={'city_clean': 'city'})
    return df

print("3. Estandarizando ciudades...")
df_clean_ciudades = estandarizar_ciudades(df_clean_precios)
registrar_metrica(2, "Estandarizacion / Filtro Ciudades", df_clean_precios, df_clean_ciudades)

## 5. Aplicación del Filtro Temporal (Sección 4)

Asignación lógica de año para registros con fecha nula según su fuente y exclusión de registros fuera del periodo de estudio definido para el proyecto (2019-2024).

In [ ]:
def limpiar_fechas(df):
    df = df.copy()
    df['created_on'] = pd.to_datetime(df['created_on'], errors='coerce')
    df['year'] = df['created_on'].dt.year
    
    año_fuente = {
        'A1_Properati': 2020, 'A2_FincaRaiz_Kaggle': 2023, 'A3_Kaggle': 2022,
        'A4_Bogota_Kaggle': 2022, 'A5_Medellin_Kaggle': 2023, 'A6_Bogota2023_Kaggle': 2023,
        'A7_Scraping_Villavicencio': 2024, 'A8_CaracPreVivNueva': 2022
    }
    df['year'] = df['year'].fillna(df['fuente'].map(año_fuente)).fillna(2023).astype(int)
    df = df[(df['year'] >= 2019) & (df['year'] <= 2024)]
    return df

print("4. Aplicando filtro temporal...")
df_clean_temporal = limpiar_fechas(df_clean_ciudades)
registrar_metrica(3, "Restriccion Temporal 2019-2024", df_clean_ciudades, df_clean_temporal)

## 6. Estandarización de Tipo de Inmueble (Sección 5)

Mapeo de categorías de propiedad en los tipos canónicos de estudio del proyecto: Casa y Apartamento.

In [ ]:
MAPA_PROPIEDADES = {
    'apartamento': 'Apartamento', 'apto': 'Apartamento', 'apartment': 'Apartamento',
    'casa': 'Casa', 'house': 'Casa', 'casa lote': 'Casa',
    'casa con conjunto cerrado': 'Casa', 'casa en conjunto cerrado': 'Casa'
}

def estandarizar_propiedad(df):
    df = df.copy()
    df['prop_raw'] = df['property_type'].astype(str).str.lower().str.strip()
    df['property_type_clean'] = df['prop_raw'].map(MAPA_PROPIEDADES)
    df = df[df['property_type_clean'].notnull()].copy()
    df = df.drop(columns=['property_type', 'prop_raw']).rename(columns={'property_type_clean': 'property_type'})
    return df

print("5. Estandarizando tipos de propiedad...")
df_clean_prop = estandarizar_propiedad(df_clean_temporal)
registrar_metrica(4, "Tipo de Inmueble (Casa/Apto)", df_clean_temporal, df_clean_prop)

## 7. Eliminación de Outliers por Grupo de Mercado (Sección 6)

Remoción de registros atípicos (outliers) utilizando cuantiles específicos de precio y área calculados dinámicamente por cada subgrupo `(city, year, property_type)`.

In [ ]:
def eliminar_outliers_grupos(df):
    df = df.copy()
    df_limpio = []
    grupos = df.groupby(['city', 'year', 'property_type'])
    
    for name, group in grupos:
        if len(group) < 10:
            df_limpio.append(group)
            continue
            
        q1_p = group['price'].quantile(0.025)
        q3_p = group['price'].quantile(0.975)
        
        group_f = group[(group['price'] >= q1_p) & (group['price'] <= q3_p)]
        
        if group_f['area'].notnull().any():
            q1_a = group_f['area'].quantile(0.01)
            q3_a = group_f['area'].quantile(0.99)
            area_null_mask = group_f['area'].isnull()
            area_valid_mask = (group_f['area'] >= q1_a) & (group_f['area'] <= q3_a)
            group_f = group_f[area_null_mask | area_valid_mask]
            
        df_limpio.append(group_f)
        
    return pd.concat(df_limpio, ignore_index=True)

print("6. Eliminando outliers por grupo...")
df_clean_outliers = eliminar_outliers_grupos(df_clean_prop)
registrar_metrica(5, "Filtro IQR Outliers por Grupo", df_clean_prop, df_clean_outliers)
diagnostico_por_fuente(df_clean_outliers, "Tras outliers IQR")

## 8. Pre-imputación de Área antes de Deduplicación (Sección 6.5)

Imputación local de áreas faltantes utilizando la mediana de grupo `(city, year, property_type)`. Esto garantiza que los registros de Properati (A1) que carecen originalmente de columna de área no compartan la clave vacía `-1` y colapsen masivamente durante el paso de deduplicación.

In [ ]:
def pre_imputar_area(df):
    """
    Imputa area nula con la mediana del grupo (city, year, property_type)
    antes de la deduplicación, para que la clave sea más precisa y no colapsen
    registros legítimamente distintos que solo carecen de área.
    """
    df = df.copy()
    mediana_grupo = df.groupby(['city', 'year', 'property_type'])['area'].transform('median')
    mediana_tipo  = df.groupby('property_type')['area'].transform('median')
    df['area'] = df['area'].fillna(mediana_grupo).fillna(mediana_tipo)
    nulos_rest = df['area'].isna().sum()
    print(f"  Nulos restantes en area tras pre-imputación: {nulos_rest:,}")
    return df

print("6.5. Pre-imputando área para mejorar calidad de la clave de deduplicación...")
df_clean_outliers = pre_imputar_area(df_clean_outliers)

## 9. Deduplicación del Dataset Consolidado (Sección 7)

Remoción de registros duplicados intra e inter-dataset. Para evitar el colapso masivo de datos en fuentes sin área, se incorporan las coordenadas geográficas (`lat` y `lon` redondeadas a 3 decimales, ~110m) en la clave de deduplicación, logrando recuperar la distribución representativa de la muestra.

In [ ]:
def eliminar_duplicados_v2(df):
    """
    Deduplicación inter-dataset mejorada usando coordenadas para evitar el colapso masivo.
    - Registros CON área: clave = city + price(1M) + area(m²) + type + year + lat(3dec) + lon(3dec)
    - Registros SIN área: clave = city + price(5M) + type + year + lat(3dec) + lon(3dec)
    """
    df = df.copy()

    prioridad_fuente = {
        'A7_Scraping_Villavicencio': 1, 'A2_FincaRaiz_Kaggle': 2,
        'A1_Properati': 3, 'A6_Bogota2023_Kaggle': 4,
        'A5_Medellin_Kaggle': 5, 'A4_Bogota_Kaggle': 6,
        'A3_Kaggle': 7, 'A8_CaracPreVivNueva': 8
    }
    df['fuente_priority'] = df['fuente'].map(prioridad_fuente).fillna(10)

    # Redondear coordenadas para la clave (los nulos se agrupan en -1.0)
    df['lat_key'] = df['lat'].round(3).fillna(-1.0)
    df['lon_key'] = df['lon'].round(3).fillna(-1.0)

    mask_con_area = df['area'].notnull()
    df_con_area  = df[mask_con_area].copy()
    df_sin_area  = df[~mask_con_area].copy()

    # Grupo 1: registros con área conocida — clave precisa
    df_con_area['dup_key'] = (
        df_con_area['city'].astype(str) + "_" +
        np.round(df_con_area['price'] / 1_000_000).astype(str) + "_" +
        np.round(df_con_area['area']).astype(str) + "_" +
        df_con_area['property_type'].astype(str) + "_" +
        df_con_area['year'].astype(str) + "_" +
        df_con_area['lat_key'].astype(str) + "_" +
        df_con_area['lon_key'].astype(str)
    )
    df_con_area = df_con_area.sort_values('fuente_priority')
    df_con_area = df_con_area.drop_duplicates(subset=['dup_key'], keep='first')

    # Grupo 2: registros sin área — clave más amplia para no sobre-colapsar
    if len(df_sin_area) > 0:
        df_sin_area['dup_key'] = (
            df_sin_area['city'].astype(str) + "_" +
            (np.round(df_sin_area['price'] / 5_000_000) * 5).astype(str) + "_" +
            df_sin_area['property_type'].astype(str) + "_" +
            df_sin_area['year'].astype(str) + "_" +
            df_sin_area['lat_key'].astype(str) + "_" +
            df_sin_area['lon_key'].astype(str)
        )
        df_sin_area = df_sin_area.sort_values('fuente_priority')
        df_sin_area = df_sin_area.drop_duplicates(subset=['dup_key'], keep='first')

    df_resultado = pd.concat([df_con_area, df_sin_area], ignore_index=True)
    df_resultado = df_resultado.drop(columns=['dup_key', 'fuente_priority', 'lat_key', 'lon_key'])

    print(f"  Con área: {len(df_con_area):,} | Sin área: {len(df_sin_area):,} | Total: {len(df_resultado):,}")
    return df_resultado

print("7. Deduplicando registros (v2 — sin area.fillna(-1))...")
df_clean_final = eliminar_duplicados_v2(df_clean_outliers)
registrar_metrica(6, "Deduplicacion Inter-Dataset v2", df_clean_outliers, df_clean_final)
diagnostico_por_fuente(df_clean_final, "Tras deduplicación v2")

## 10. Imputación de Valores Faltantes (Sección 8)

Imputación de nulos remanentes en área, cuartos, baños y estrato socioeconómico usando las medianas de grupo. Imputación de coordenadas geográficas faltantes (`lat`/`lon`) mediante el centroide oficial de cada ciudad.

In [ ]:
def imputar_valores_faltantes(df):
    df = df.copy()
    
    mediana_area_grupo = df.groupby(['city', 'year', 'property_type'])['area'].transform('median')
    df['area'] = df['area'].fillna(mediana_area_grupo)
    mediana_area_global = df.groupby('property_type')['area'].transform('median')
    df['area'] = df['area'].fillna(mediana_area_global)
    
    mediana_hab_grupo = df.groupby(['city', 'property_type'])['rooms'].transform('median')
    df['rooms'] = df['rooms'].fillna(mediana_hab_grupo).fillna(3).astype(int)
    df['rooms'] = df['rooms'].clip(lower=1)
    
    mediana_ban_grupo = df.groupby(['city', 'property_type'])['bathrooms'].transform('median')
    df['bathrooms'] = df['bathrooms'].fillna(mediana_ban_grupo).fillna(2).astype(int)
    df['bathrooms'] = df['bathrooms'].clip(lower=1)
    
    if 'barrio' in df.columns:
        mediana_estrato_barrio = df.groupby(['city', 'barrio'])['estrato'].transform('median')
        df['estrato'] = df['estrato'].fillna(mediana_estrato_barrio)
        
    mediana_estrato_ciudad = df.groupby('city')['estrato'].transform('median')
    df['estrato'] = df['estrato'].fillna(mediana_estrato_ciudad).fillna(3).astype(int)
    df['estrato'] = df['estrato'].clip(1, 6).astype(int)
    
    # Imputar lat/lon faltantes con el centroide de la ciudad
    centroides = {
        'Bogotá': (4.6097, -74.0817), 'Medellín': (6.2442, -75.5812),
        'Cali': (3.4516, -76.5320), 'Barranquilla': (10.9685, -74.7813),
        'Cartagena': (10.3997, -75.4763), 'Bucaramanga': (7.1254, -73.1198),
        'Pereira': (4.8133, -75.6961), 'Manizales': (5.0689, -75.5174),
        'Armenia': (4.5339, -75.6811), 'Cúcuta': (7.8939, -72.5078),
        'Ibagué': (4.4389, -75.2322), 'Villavicencio': (4.1420, -73.6266)
    }
    
    if 'lat' in df.columns and 'lon' in df.columns:
        df['lat'] = df.apply(lambda row: centroides.get(row['city'], (np.nan, np.nan))[0] if pd.isna(row['lat']) else row['lat'], axis=1)
        df['lon'] = df.apply(lambda row: centroides.get(row['city'], (np.nan, np.nan))[1] if pd.isna(row['lon']) else row['lon'], axis=1)
    
    return df

print("8. Imputando valores faltantes...")
df_imputado = imputar_valores_faltantes(df_clean_final)

## 11. Integración de Datos Macroeconómicos (Sección 9)

Carga y agregación anual de variables macroeconómicas del Grupo B (salario mínimo mensual, IPC base 2018, tasas hipotecarias, variaciones oficiales de los índices de vivienda IPVU e IPVN del DANE y tasa de desempleo regional de las 13 principales ciudades o fallback nacional).

In [ ]:
def cargar_e_integrar_macro(df_inmuebles):
    # B3: Salario
    b3 = pd.read_csv(os.path.join(DIR_RAW, "B3_salario_minimo_historico.csv"), encoding='utf-8-sig')
    b3 = b3.rename(columns={'Ano': 'year', 'Salario_minimo_mensual': 'salario_mensual'})[['year', 'salario_mensual']]

    # B4: IPC
    b4 = pd.read_csv(os.path.join(DIR_RAW, "B4_ipc_colombia_anual.csv"), encoding='utf-8-sig')
    b4 = b4.rename(columns={'Ano': 'year', 'Variacion_IPC_%': 'ipc_var_anual'})
    b4 = b4.sort_values('year').reset_index(drop=True)
    b4['ipc_base2018'] = 100.0
    idx_2018 = b4.index[b4['year'] == 2018].tolist()
    if idx_2018:
        idx = idx_2018[0]
        for i in range(idx+1, len(b4)):
            b4.loc[i, 'ipc_base2018'] = b4.loc[i-1, 'ipc_base2018'] * (1 + b4.loc[i, 'ipc_var_anual']/100)
        for i in range(idx-1, -1, -1):
            b4.loc[i, 'ipc_base2018'] = b4.loc[i+1, 'ipc_base2018'] / (1 + b4.loc[i+1, 'ipc_var_anual']/100)

    # B2: Tasa Hipotecaria
    b2 = pd.read_csv(os.path.join(DIR_RAW, "B2_tasa_hipotecaria_semanal.csv"), encoding='latin1')
    col_fecha = b2.columns[0]
    b2['year'] = pd.to_datetime(b2[col_fecha], errors='coerce').dt.year
    col_hipotecaria = [c for c in b2.columns if 'colocaci' in c.lower()][0]
    b2_year = b2.groupby('year')[col_hipotecaria].mean().reset_index()
    b2_year = b2_year.rename(columns={col_hipotecaria: 'tasa_hipotecaria_anual'})

    # B5: Desempleo (B5 mejorado: merge 13 ciudades y nacional)
    b5 = pd.read_csv(os.path.join(DIR_RAW, "B5_geih_empleo_colombia.csv"), encoding='utf-8-sig')
    b5['year'] = pd.to_datetime(b5['fecha'], errors='coerce').dt.year
    b5_13 = b5[b5['grupo'] == '13_ciudades'].groupby('year')['td'].mean().reset_index().rename(columns={'td': 'tasa_desempleo_13'})
    b5_nac = b5[b5['grupo'] == 'nacional'].groupby('year')['td'].mean().reset_index().rename(columns={'td': 'tasa_desempleo_nac'})

    # B1: Indices IPVU e IPVN
    b1 = pd.read_csv(os.path.join(DIR_RAW, "B1_indices_precios_vivienda.csv"), encoding='utf-8-sig')
    b1.columns = [c.strip() for c in b1.columns]
    b1['year'] = pd.to_datetime(b1['fecha'], errors='coerce').dt.year
    b1_year = b1.groupby('year')[['ipvnbr_índice_nominal_agregado', 'ipvu_indice_nominal']].mean().reset_index()
    b1_year['ipvn_variacion_anual'] = b1_year['ipvnbr_índice_nominal_agregado'].pct_change() * 100
    b1_year['ipvu_variacion_anual'] = b1_year['ipvu_indice_nominal'].pct_change() * 100

    # Merge todo
    df_macro = pd.DataFrame({'year': range(2019, 2025)})
    df_macro = df_macro.merge(b3, on='year', how='left')
    df_macro = df_macro.merge(b4, on='year', how='left')
    df_macro = df_macro.merge(b2_year, on='year', how='left')
    df_macro = df_macro.merge(b5_13, on='year', how='left')
    df_macro = df_macro.merge(b5_nac, on='year', how='left')
    df_macro['tasa_desempleo'] = df_macro['tasa_desempleo_13'].fillna(df_macro['tasa_desempleo_nac'])
    df_macro = df_macro.drop(columns=['tasa_desempleo_13', 'tasa_desempleo_nac'])
    df_macro = df_macro.merge(b1_year[['year', 'ipvn_variacion_anual', 'ipvu_variacion_anual']], on='year', how='left')
    
    # Rellenar valores nulos (para evitar que se propaguen)
    df_macro = df_macro.ffill().bfill()
    
    # Merge vivienda + tabla macro general
    df_fusionado = df_inmuebles.merge(df_macro, on='year', how='left')
    
    return df_fusionado, df_macro

print("9. Integrando datos macroeconómicos...")
df_integrado_macro, df_tabla_macro = cargar_e_integrar_macro(df_imputado)

## 12. Construcción de Variables Derivadas (Sección 10)

Cálculo de métricas derivadas del mercado inmobiliario y asequibilidad: Salario Anual, Índice de Asequibilidad de Vivienda (IAH), Precio Real deflactado a base 2018, Precio por m², Cuota de Crédito Hipotecario Estimada y Nivel Cualitativo de Asequibilidad.

In [ ]:
def calcular_cuota_mensual(precio, tasa_anual, meses=180, financia=0.70):
    if pd.isna(precio) or pd.isna(tasa_anual) or tasa_anual <= 0:
        return np.nan
    monto_credito = precio * financia
    tasa_mensual = (1 + (tasa_anual / 100)) ** (1/12) - 1
    cuota = monto_credito * (tasa_mensual * (1 + tasa_mensual)**meses) / ((1 + tasa_mensual)**meses - 1)
    return cuota

def construir_variables_derivadas(df):
    df = df.copy()
    df['salario_anual'] = df['salario_mensual'] * 12
    df['IAH'] = df['price'] / df['salario_anual']
    df['precio_real'] = df['price'] / (df['ipc_base2018'] / 100)
    df['precio_m2'] = df['price'] / df['area']
    
    df['cuota_mensual'] = df.apply(
        lambda row: calcular_cuota_mensual(row['price'], row['tasa_hipotecaria_anual']), axis=1
    )
    
    df['ratio_cuota_salario'] = df['cuota_mensual'] / df['salario_mensual']
    
    condiciones = [
        (df['IAH'] <= 5),
        (df['IAH'] > 5) & (df['IAH'] <= 10),
        (df['IAH'] > 10) & (df['IAH'] <= 20),
        (df['IAH'] > 20)
    ]
    categorias = ['Accesible', 'Moderado', 'Elevado', 'Crítico']
    df['nivel_accesibilidad'] = np.select(condiciones, categorias, default='Crítico')
    
    return df

print("10. Construyendo variables derivadas...")
df_variables = construir_variables_derivadas(df_integrado_macro)

## 13. Validación y Control de Integridad (Sección 11)

Aserciones lógicas e integridad estructural para garantizar que no queden valores nulos en variables críticas de modelamiento ni anomalías físicas.

In [ ]:
def validar_dataset_final(df):
    null_counts = df.isnull().sum()
    print("Conteo de nulos por columna:")
    print(null_counts[null_counts > 0])
    
    columnas_criticas = ['price', 'area', 'rooms', 'bathrooms', 'property_type', 'city', 'estrato', 'year', 'IAH']
    for col in columnas_criticas:
        assert df[col].isnull().sum() == 0, f"Error: Existen valores nulos en la columna crítica {col}"
        
    assert (df['price'] > 0).all(), "Error: Existen precios menores o iguales a cero"
    assert (df['area'] > 0).all(), "Error: Existen areas menores o iguales a cero"
    assert (df['rooms'] >= 1).all(), "Error: Cantidad de habitaciones invalida"
    assert (df['bathrooms'] >= 1).all(), "Error: Cantidad de banos invalida"
    assert df['city'].isin(MAPA_CIUDADES.values()).all(), "Error: Ciudades fuera del catalogo"
    assert df['year'].between(2019, 2024).all(), "Error: Anos fuera del periodo temporal"
    assert df['estrato'].between(1, 6).all(), "Error: Estratos fuera del rango 1-6"
    print("Validacion de integridad aprobada exitosamente.")

print("11. Validando dataset final...")
columns_to_export = [
    'price', 'area', 'rooms', 'bathrooms', 'property_type', 'city', 'lat', 'lon', 
    'created_on', 'estrato', 'fuente', 'year', 'salario_mensual', 'ipc_var_anual', 
    'ipc_base2018', 'tasa_hipotecaria_anual', 'tasa_desempleo', 'ipvu_variacion_anual', 
    'ipvn_variacion_anual', 'salario_anual', 'IAH', 'precio_real', 'precio_m2', 
    'cuota_mensual', 'ratio_cuota_salario', 'nivel_accesibilidad'
]
df_final = df_variables[columns_to_export].copy()
validar_dataset_final(df_final)

## 14. Validación Cruzada con IPVN DANE

Cálculo comparativo de la variación promedio del precio por metro cuadrado frente a la variación oficial del Índice de Precios de Vivienda Nueva (IPVN) publicado por el DANE.

In [ ]:
def realizar_validacion_ipvn(df):
    print("\n=== Validación Cruzada con IPVN DANE ===")
    df_valid = df[df['city'].isin(['Bogotá', 'Medellín'])].copy()
    
    # Calcular precio_m2 promedio por ciudad y año
    df_m2 = df_valid.groupby(['city', 'year'])['precio_m2'].mean().reset_index()
    
    # Calcular variación anual propia
    df_m2['variacion_precio_m2_%'] = df_m2.groupby('city')['precio_m2'].pct_change() * 100
    
    # Merge con la variación oficial del IPVN
    df_ipvn_oficial = df.groupby('year')['ipvn_variacion_anual'].first().reset_index()
    df_comparativo = df_m2.merge(df_ipvn_oficial, on='year', how='left')
    
    print(df_comparativo[['city', 'year', 'precio_m2', 'variacion_precio_m2_%', 'ipvn_variacion_anual']].dropna().to_string(index=False))
    
    print("\nDiferencia Promedio vs IPVN Oficial (Bogotá y Medellín):")
    for city in ['Bogotá', 'Medellín']:
        sub = df_comparativo[(df_comparativo['city'] == city) & (df_comparativo['variacion_precio_m2_%'].notnull())]
        diff = np.abs(sub['variacion_precio_m2_%'] - sub['ipvn_variacion_anual']).mean()
        print(f"  - {city}: {diff:.3f} pp de diferencia promedio")

realizar_validacion_ipvn(df_final)

## 15. Exportación del Dataset Limpio (Sección 12)

Escritura del archivo consolidado `vivienda_colombia_limpio.csv` (con codificación UTF-8 con BOM para su compatibilidad en hojas de cálculo como Microsoft Excel) y del embudo metodológico de métricas a `reporte_limpieza.csv`.

In [ ]:
print("12. Exportando...")
df_final.to_csv(os.path.join(DIR_PROCESSED, "vivienda_colombia_limpio.csv"), index=False, encoding='utf-8-sig')
pd.DataFrame(reporte_metricas).to_csv(os.path.join(DIR_PROCESSED, "reporte_limpieza.csv"), index=False, encoding='utf-8-sig')
print("Archivos exportados exitosamente.")